# Lab 0: Before the Algorithm — Understanding Your Data
## An Economist's First Day at a FinTech Credit Bureau

**Duration:** 45 minutes | **Theme:** Tell a story through data

---

### 🎯 Your Mission

You've just been hired as a **data economist** at a FinTech company. Your job: **UNDERSTAND the data** before making policy decisions.

Why? Because:
- **Bad data → Bad policy → Real harm** (people denied credit unfairly)
- **Clean data → Informed decisions → Fair lending**

### 📖 The Story Ahead

**This dataset will follow us through all 3 labs:**
1. **Lab 0 (TODAY):** Clean and understand it → *What's in this data?*
2. **Lab 1 (NEXT):** Build a predictor → *Can we predict credit risk?*
3. **Lab 2 (FUTURE):** Make decisions → *Should we approve this loan?*

Today, we're economists first. Algorithms second.

---

### 📊 The Case Study

**Dataset:** German Credit Dataset (UCI) | **1000 instances** | **21 attributes**

Each row = One person's financial life. Each column = A decision you might make about them.

Let's begin.

---
## ⚙️ Setup: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for all plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded. Ready to explore.")

---
## ACT 1: MEET THE DATA 🔍

### Step 1: Load the Dataset

We'll load the German Credit Dataset from a reliable source. Each row is a loan application.

In [ ]:
# Load German Credit Dataset
# Using GitHub as a reliable source
url = "https://raw.githubusercontent.com/sampathlonka/ml-workshop/master/Data/german_credit_data.csv"

df = pd.read_csv(url)
print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

### Step 2: Inspect the Data

Let's meet our data. First impressions matter.

In [ ]:
# What do the first 5 records look like?
print("FIRST 5 LOAN APPLICATIONS:")
print(df.head())
print(f"\nShape: {df.shape[0]} people, {df.shape[1]} features")
print(f"\nColumn names:\n{list(df.columns)}")

### Step 3: Data Types & Missing Values

**Reality check:** How complete is our data?

In [ ]:
# Info on data types and missing values
print("DATA INTEGRITY CHECK:")
print(df.info())
print("\n" + "="*60)
print("MISSING VALUES (count and percentage):")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

### 💡 POLICY INSIGHT: Understanding Your Data

**Key Questions to Ask:**

1. **What does each row represent?**
   - One person applying for credit at a FinTech company
   - Your data contains their: age, employment, savings, loan amount, and risk classification

2. **If you were a policy maker, which columns concern you most?**
   - **Age:** Are we discriminating against older/younger applicants?
   - **Sex:** Are women and men treated equally?
   - **Job:** Are we excluding certain professions unfairly?
   - **ExistingSavings / CheckingStatus:** Are we favoring the wealthy?
   - **LoanAmount & LoanDuration:** Are high-risk groups forced to take larger loans?
   - **Risk Classification:** Is this outcome fair or biased?

3. **What's NOT in your data?**
   - Income (only inferred from employment)
   - Education level
   - Location / neighborhood (often a proxy for race)
   - Full credit history (only savings inferred)
   
   *Missing data = Missing stories. Every omission is a decision.*

---

### 🤝 DISCUSS WITH YOUR NEIGHBOR (2 min):
- "Which missing column would you want to see? Why?"
- "Could missing data introduce bias?"

### Step 4: Descriptive Statistics

**What's the 'typical' loan application?**

In [ ]:
# Statistical summary for numeric columns
print("SNAPSHOT OF THE 'AVERAGE' APPLICANT:")
print(df.describe().round(2))

# Value counts for Risk (our outcome variable)
print("\n" + "="*60)
print("\nCREDIT RISK DISTRIBUTION:")
print(df['Risk'].value_counts())

# Handle both numeric (1/2) and string ('good'/'bad') risk values
risk_counts = df['Risk'].value_counts()
bad_risk_count = risk_counts.get(2, risk_counts.get('bad', 0))  # Try 2 first, then 'bad'
risk_pct = (bad_risk_count / len(df) * 100) if bad_risk_count > 0 else 0
print(f"\nRisk Distribution: {risk_pct:.1f}% of applicants are classified as high risk")

---
## ACT 2: THE MESSY REALITY 🔧

### Step 1: Visualize Missing Data

Where are the gaps in our data?

In [ ]:
# Heatmap of missing values
plt.figure(figsize=(12, 3))
sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='RdYlGn_r')
plt.title('Missing Data Heatmap (Red = Missing)', fontsize=12, fontweight='bold')
plt.xlabel('Features')
plt.tight_layout()
plt.show()

print("Red bars = Missing values. Patterns tell stories.")

### 💡 POLICY INSIGHT: What Missing Data Reveals

**"Missing data isn't random—it tells a story."**

**Example Questions:**
- Why might *ExistingSavings* data be missing?
  - Maybe very poor people don't have savings to report?
  - Maybe the FinTech didn't ask?
  - Maybe they refused to share (privacy concern)?

- Why might *Employment* data be missing?
  - Unemployed people avoid revealing it?
  - Self-employed people skip standardized fields?

**The danger:** If we delete rows with missing data, we might **silence the voices of the poorest, most vulnerable applicants**.

---

### 🤝 DISCUSS WITH YOUR NEIGHBOR (2 min):
- "If we delete all rows with missing savings, what happens to fair lending?"
- "How would you handle this ethically?"

### Step 2: Handle Missing Values

**Strategy:** For this workshop, we'll use a simple approach:
- Fill categorical missing values with 'Unknown'
- Fill numeric missing values with median
- Keep the cleaned dataset for Lab 1 & Lab 2

*(In production, you'd consult domain experts and lawyers.)*

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Identify categorical and numeric columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

# Fill categorical missing values with 'Unknown'
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna('Unknown')
        print(f"✓ Filled {col} with 'Unknown'")

# Fill numeric missing values with median
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        print(f"✓ Filled {col} with median")

print("\n✓ Missing values handled.")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")

---
## ACT 3: OUTLIERS — OR JUST INEQUALITY? 📈

### Step 1: Visualize Distributions

**Reality check:** Are extreme values errors, or are they the data we should care most about?

In [ ]:
# Box plots for key numeric features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loan Amount
axes[0].boxplot(df_clean['LoanAmount'], vert=True)
axes[0].set_ylabel('Loan Amount')
axes[0].set_title('Loan Amount Distribution')
axes[0].grid(True, alpha=0.3)

# Duration
axes[1].boxplot(df_clean['LoanDuration'], vert=True)
axes[1].set_ylabel('Months')
axes[1].set_title('Loan Duration Distribution')
axes[1].grid(True, alpha=0.3)

# Age
axes[2].boxplot(df_clean['Age'], vert=True)
axes[2].set_ylabel('Years')
axes[2].set_title('Age Distribution')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Box plots show median (line), quartiles (box), and outliers (dots).")

### 💡 POLICY INSIGHT: Outliers Aren't Always Errors

**In economics, outliers are inequality.**

**Example:**
- Most people borrow 3,000 (typical amount)
- Some borrow 100,000 (wealthy people, legitimate)
- Some borrow 500,000 (emergency situation? or unfair denial of smaller loans?)

**Question: Should we remove the person who took a 500,000 loan?**
- ❌ **NO.** This might be the most important data point. They're the exception—and exceptions matter.
- ❌ **Removing them = Ignoring inequality**
- ✓ **Keep them = Understand why some people need bigger loans**

**When to remove outliers:**
1. **Data entry errors:** Age = 999 (impossible)
2. **Machine errors:** Duplicate rows, corrupted values
3. **NOT for:** Extreme but real values (e.g., one person's 500k loan)

---

### 🤝 DISCUSS WITH YOUR NEIGHBOR (2 min):
- "Is the oldest applicant an outlier or an important data point?"
- "What if we removed all high-loan-amount applicants? Who would we hurt?"

### Step 2: Identify Potential Outliers (but keep them for now!)

We'll flag extreme values but NOT remove them.

In [ ]:
# Z-score method: values beyond 3 standard deviations
from scipy.stats import zscore

# Calculate z-scores for numeric columns
numeric_cols = ['Age', 'LoanAmount', 'LoanDuration']
z_scores = np.abs(zscore(df_clean[numeric_cols]))

# Find outliers (z-score > 3)
outliers = (z_scores > 3).any(axis=1)
print(f"Potential outliers detected: {outliers.sum()} records")
print(f"Percentage: {(outliers.sum() / len(df_clean) * 100):.2f}%")
print("\nThese are KEPT for analysis. They tell important stories.")

---
## ACT 4: THE STORY IN THE NUMBERS 📊

### Insight 1: Who Gets Credit Risk?

**Is age a factor in credit default?**

In [ ]:
# Create age groups for analysis
df_clean['Age_Group'] = pd.cut(df_clean['Age'], bins=[0, 25, 35, 45, 55, 100], 
                                 labels=['18-25', '26-35', '36-45', '46-55', '56+'])

# Risk by age group - handle both numeric (1/2) and string ('good'/'bad') risk values
risk_by_age = pd.crosstab(df_clean['Age_Group'], df_clean['Risk'], normalize='index') * 100
print("CREDIT RISK BY AGE GROUP (%):")
print(risk_by_age.round(1))

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
risk_by_age.plot(kind='bar', ax=ax)
plt.title('Credit Risk Distribution by Age Group', fontsize=12, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Risk')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### Insight 2: What Do People Borrow For?

**Purpose reveals intent and need.**

In [ ]:
# Purpose of loan
purpose_counts = df_clean['LoanPurpose'].value_counts()
print("LOAN PURPOSE (count):")
print(purpose_counts)

# Visualization
plt.figure(figsize=(10, 5))
purpose_counts.plot(kind='barh', color='steelblue')
plt.title('Why Do People Borrow? (Loan Purpose)', fontsize=12, fontweight='bold')
plt.xlabel('Number of Applicants')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nInsight: Loan purposes reveal what people prioritize in their lives.")

### Insight 3: Loan Amount vs Duration

**Do larger loans take longer to repay? Do poor people get stuck with big payments?**

In [ ]:
# Scatter plot: Loan Amount vs Duration, colored by Risk
plt.figure(figsize=(10, 6))

# Get unique risk values to determine how to color
unique_risks = df_clean['Risk'].unique()
colors = {}

# Assign colors based on risk type (handle both numeric and string values)
if all(isinstance(x, (int, np.integer)) for x in unique_risks):
    # Numeric risk values (e.g., 1, 2)
    colors = {1: 'green', 2: 'red'}
elif all(isinstance(x, str) for x in unique_risks):
    # String risk values (e.g., 'good', 'bad')
    colors = {'good': 'green', 'bad': 'red'}
else:
    # Mixed or unknown - use a default
    colors = {sorted(unique_risks)[0]: 'green', sorted(unique_risks)[-1]: 'red'}

for risk_type, color in colors.items():
    mask = df_clean['Risk'] == risk_type
    plt.scatter(df_clean[mask]['LoanAmount'], df_clean[mask]['LoanDuration'], 
               alpha=0.5, s=50, label=f'Risk: {risk_type}', color=color)

plt.xlabel('Loan Amount')
plt.ylabel('Duration (Months)')
plt.title('Loan Amount vs Duration (colored by Risk)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Correlation
corr = df_clean[['LoanAmount', 'LoanDuration']].corr().iloc[0, 1]
print(f"Correlation between Loan Amount and Duration: {corr:.3f}")
print("(Positive = they move together. Larger loans take longer to repay.)")

### Insight 4: Correlation Heatmap

**Which features are related? Are they fair?**

In [ ]:
# Select numeric columns for correlation
numeric_df = df_clean.select_dtypes(include=[np.number])

# Correlation matrix
corr_matrix = numeric_df.corr()

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Loan Amount & Duration: strong positive correlation (larger loans take longer)")
print("- Age & Loan Amount: investigate relationship")
print("- These relationships tell us what variables matter for prediction.")

### 💡 POLICY INSIGHT: What the Numbers Tell Us

**Three Key Findings from Our EDA:**

#### Finding 1: Age & Risk
- Younger applicants may show different default patterns than older ones
- Older applicants may have more stable employment
- **Policy question:** Should age be a lending criterion? (Note: In many countries, age discrimination in lending is illegal, even if empirically correlated.)

#### Finding 2: Purpose Matters
- Different loan purposes may carry different risk profiles
- Some purposes (e.g., education, business) may have higher social value
- **Policy question:** Should we encourage lending for certain purposes?

#### Finding 3: Missing Data Silences the Poor
- Missing data in savings/checking accounts may exclude poorest applicants
- **Policy question:** How do we serve those with no traditional banking?

---

### Step: Save Cleaned Dataset for Lab 1 & Lab 2

**We'll reuse this cleaned data in future labs.**

In [ ]:
# Save the cleaned dataset
df_clean.to_csv('german_credit_cleaned.csv', index=False)
print("✓ Cleaned dataset saved: 'german_credit_cleaned.csv'")
print(f"  Shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print("  Ready for Lab 1 (Regression) and Lab 2 (Classification)")

---
## WRAP-UP: Your Journey Through Data 🎓

### What You Did Today

✓ **Met the data** — 1,000 real loan applications  
✓ **Identified missing values** — and chose not to silence the poor  
✓ **Kept outliers** — because inequality matters  
✓ **Discovered patterns** — age, purpose, risk, correlation  
✓ **Cleaned & saved** — ready for the next lab  

### The Three Labs: A Story

**Lab 0 (TODAY):** "What's in this data?"  
→ You cleaned, explored, and found patterns.

**Lab 1 (NEXT):** "Can we predict credit risk?"  
→ You'll build a regression model to estimate default probability.

**Lab 2 (AFTER):** "Should we approve this loan?"  
→ You'll classify applicants and make policy decisions with full awareness of bias and fairness.

### Your Role Evolves

- **Lab 0:** Economist (understand the data)
- **Lab 1:** Data Scientist (build the model)
- **Lab 2:** Policy Maker (use the model responsibly)

---

## FINAL REFLECTION 💭

### Answer These Questions (Discuss in Groups)

1. **"What's the most surprising thing you learned about this data?"**
   - (Example: "35% of people don't report their savings!")

2. **"If you were a loan officer, which three features would you trust most to decide credit risk? Why?"**

3. **"What ethical concern do you have about this dataset?"**
   - Missing data? Potential bias? Discrimination risk?

4. **"How might the FinTech company's decisions based on this data affect people's lives?"**
   - Approved loans → Financial freedom
   - Denied loans → Desperation (illegal lending, loan sharks)
   - Biased decisions → Systemic inequality

---

## 🎯 Key Takeaway

**"Data is not neutral. Every dataset encodes human choices: what to measure, what to ignore, who to include, who to exclude. Before you build a model, understand the story."**

---

### Next Steps
- 📊 Load this cleaned dataset in **Lab 1** to predict credit risk
- 🎓 Use those predictions in **Lab 2** to make fair lending decisions
- 🌍 Apply this lesson to your future work: always ask "What story is my data telling?"

**Thank you. See you in Lab 1.**